# 01 — Recolección y verificación de datos
**Proyecto:** Recomendador de draft de Brawl Stars con reconocimiento visual de brawlers
**Objetivo de este notebook:** comprobar que las dos fuentes funcionan y documentar exactamente qué contienen. **No entrenamos ningún modelo aquí.**

Preguntas que responde:
1. ¿Cuántos brawlers devuelve BrawlAPI?
2. ¿Qué resolución y formato tienen sus imágenes?
3. ¿Qué columnas tiene el dataset de partidas Ranked?
4. ¿Cómo están representados equipos, mapas, modos y resultados?
5. ¿Qué volumen de registros podemos procesar en Colab sin sobrecargarlo?

Al final hay una tabla resumen para completar el informe.

## 0. Instalación de dependencias

In [1]:
!pip -q install datasets pillow requests pandas

## 1. Configuración e imports
Las constantes están centralizadas aquí para que el notebook sea fácil de reproducir.

In [2]:
import io
import itertools
import time

import requests
import pandas as pd
from PIL import Image

BRAWLAPI_URL = "https://api.brawlapi.com/v1/brawlers"
DATASET_ID = "EliF77/brawlstars-ranked"
TIMEOUT = 20          # segundos por petición HTTP
SAMPLE_SIZE = 1000    # filas para inspeccionar el esquema sin descargar todo
FULL_SCAN = False     # True = recorrer el dataset completo (lento; úsalo solo si lo necesitas)

## 2. BrawlAPI: ¿cuántos brawlers hay?
Pedimos el catálogo y revisamos la **forma** de la respuesta antes de asumir nada. Algunas APIs devuelven una lista directa; otras, un diccionario con la lista dentro de una clave.

In [3]:
r = requests.get(BRAWLAPI_URL, timeout=TIMEOUT)
print("Código HTTP:", r.status_code)
r.raise_for_status()

payload = r.json()
print("Tipo de respuesta:", type(payload).__name__)

if isinstance(payload, dict):
    print("Claves de primer nivel:", list(payload.keys()))
    brawlers = payload.get("list", [])
else:
    brawlers = payload

print("Número de brawlers devueltos:", len(brawlers))
print("Campos de cada brawler:", list(brawlers[0].keys()))

Código HTTP: 200
Tipo de respuesta: dict
Claves de primer nivel: ['list']
Número de brawlers devueltos: 109
Campos de cada brawler: ['id', 'avatarId', 'name', 'hash', 'path', 'fankit', 'released', 'version', 'link', 'imageUrl', 'imageUrl2', 'imageUrl3', 'class', 'rarity', 'unlock', 'description', 'descriptionHtml', 'starPowers', 'gadgets', 'videos']


Convertimos el catálogo en una tabla. Como no sabemos aún el nombre exacto del campo de imagen, lo buscamos por palabra clave (`image`) en lugar de escribirlo a mano.

In [4]:
df_brawlers = pd.DataFrame(brawlers)
print(df_brawlers.columns.tolist())

img_cols = [c for c in df_brawlers.columns if "image" in c.lower()]
print("Campos que contienen imágenes:", img_cols)

# Si 'class' viene como diccionario, lo convertimos a texto para contar
if "class" in df_brawlers.columns:
    print("\nDistribución por clase:")
    print(df_brawlers["class"].astype(str).value_counts())

df_brawlers[["id", "name"] + [c for c in ["class"] if c in df_brawlers.columns]].head(25)

['id', 'avatarId', 'name', 'hash', 'path', 'fankit', 'released', 'version', 'link', 'imageUrl', 'imageUrl2', 'imageUrl3', 'class', 'rarity', 'unlock', 'description', 'descriptionHtml', 'starPowers', 'gadgets', 'videos']
Campos que contienen imágenes: ['imageUrl', 'imageUrl2', 'imageUrl3']

Distribución por clase:
class
{'id': 1, 'name': 'Damage Dealer'}    22
{'id': 5, 'name': 'Controller'}       21
{'id': 6, 'name': 'Assassin'}         19
{'id': 2, 'name': 'Tank'}             16
{'id': 7, 'name': 'Support'}          13
{'id': 3, 'name': 'Marksman'}         11
{'id': 4, 'name': 'Artillery'}         7
Name: count, dtype: int64


,id,name,class
0,16000110,Vince,"{'id': 1, 'name': 'Damage Dealer'}"
1,16000109,Cosmo,"{'id': 5, 'name': 'Controller'}"
2,16000108,Wendy,"{'id': 7, 'name': 'Support'}"
3,16000107,Nori,"{'id': 6, 'name': 'Assassin'}"
4,16000106,Bolt,"{'id': 2, 'name': 'Tank'}"
5,16000105,Starr Nova,"{'id': 6, 'name': 'Assassin'}"
6,16000104,Damian,"{'id': 2, 'name': 'Tank'}"
7,16000103,Najia,"{'id': 1, 'name': 'Damage Dealer'}"
8,16000102,Sirius,"{'id': 5, 'name': 'Controller'}"
9,16000101,Glowy,"{'id': 7, 'name': 'Support'}"


## 3. Imágenes: resolución, formato y accesibilidad
Descargamos una muestra de íconos y revisamos tamaño, modo de color y formato real del archivo. No asumimos PNG: lo leemos del propio archivo.

In [5]:
if not img_cols:
    raise RuntimeError("No se encontró ningún campo de imagen en BrawlAPI. Revisa la lista de campos impresa arriba.")

IMG_FIELD = img_cols[0]   # campo elegido; si hay varios, compara en la celda siguiente
print("Campo de imagen usado:", IMG_FIELD)

registros_img = []
for b in brawlers[:5]:
    url = b.get(IMG_FIELD)
    if not url:
        registros_img.append({"brawler": b.get("name"), "error": "sin URL"})
        continue
    try:
        resp = requests.get(url, timeout=TIMEOUT)
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content))
        registros_img.append({
            "brawler": b.get("name"),
            "http": resp.status_code,
            "formato": img.format,
            "modo_color": img.mode,
            "ancho": img.size[0],
            "alto": img.size[1],
            "bytes": len(resp.content),
            "content_type": resp.headers.get("Content-Type"),
        })
    except Exception as e:
        registros_img.append({"brawler": b.get("name"), "error": str(e)[:120]})

df_img = pd.DataFrame(registros_img)
df_img

Campo de imagen usado: imageUrl


,brawler,http,formato,modo_color,ancho,alto,bytes,content_type
0,Vince,200,PNG,P,200,200,8182,image/png
1,Cosmo,200,PNG,P,200,200,10045,image/png
2,Wendy,200,PNG,P,200,200,12191,image/png
3,Nori,200,PNG,P,200,200,10605,image/png
4,Bolt,200,PNG,P,200,200,6373,image/png


Si los tamaños son distintos entre brawlers, hay que normalizarlos a una resolución fija antes de extraer características. Lo registramos aquí para decidirlo con datos.

In [6]:
if "ancho" in df_img.columns:
    print("Resoluciones únicas en la muestra:",
          df_img[["ancho", "alto"]].drop_duplicates().values.tolist())
    print("Formatos únicos:", df_img["formato"].dropna().unique().tolist())
    print("Modos de color únicos:", df_img["modo_color"].dropna().unique().tolist())

Resoluciones únicas en la muestra: [[200, 200]]
Formatos únicos: ['PNG']
Modos de color únicos: ['P']


## 4. Dataset de partidas Ranked: esquema
Usamos `streaming=True` para leer solo una parte del dataset y no descargarlo entero. Así vemos las columnas sin cargar 1.7 millones de filas.

In [7]:
from datasets import load_dataset

ds_stream = load_dataset(DATASET_ID, split="train", streaming=True)

print("Features (esquema declarado):")
try:
    print(ds_stream.features)
except Exception as e:
    print("No se pudo leer el esquema declarado:", e)

sample = list(itertools.islice(ds_stream, SAMPLE_SIZE))
df_sample = pd.DataFrame(sample)
print("\nFilas en la muestra:", len(df_sample))
print("Columnas:", df_sample.columns.tolist())
print("\nTipos de datos:")
print(df_sample.dtypes)

README.md:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

Features (esquema declarado):
{'id': Value('int64'), 'battle_time': Value('string'), 'mode': Value('string'), 'map': Value('string'), 'record': Value('string'), 'star_brawler': Value('string'), 'star_power': Value('int64'), 'star_player_tag': Value('string'), 'star_elo': Value('int64'), 'avg_elo': Value('float64'), 't1_b0_name': Value('string'), 't1_b0_elo': Value('int64'), 't1_b0_power': Value('int64'), 't1_b0_tag': Value('string'), 't1_b1_name': Value('string'), 't1_b1_elo': Value('int64'), 't1_b1_power': Value('int64'), 't1_b1_tag': Value('string'), 't1_b2_name': Value('string'), 't1_b2_elo': Value('int64'), 't1_b2_power': Value('int64'), 't1_b2_tag': Value('string'), 't2_b0_name': Value('string'), 't2_b0_elo': Value('int64'), 't2_b0_power': Value('int64'), 't2_b0_tag': Value('string'), 't2_b1_name': Value('string'), 't2_b1_elo': Value('int64'), 't2_b1_power': Value('int64'), 't2_b1_tag': Value('string'), 't2_b2_name': Value('string'), 't2_b2_elo': Value('int64'), 't2_b2_power': Val

## 5. Representación de equipos, mapas, modos y resultados
Para cada columna mostramos: tipo, número de valores distintos en la muestra y ejemplos. Así vemos si los brawlers vienen como texto, como lista o como identificador numérico, y si el resultado es binario.

In [8]:
for col in df_sample.columns:
    serie = df_sample[col]
    n_unicos = serie.astype(str).nunique()
    print(f"\n--- {col} | tipo={serie.dtype} | valores únicos en muestra={n_unicos}")
    print("Ejemplos:", serie.head(3).tolist())


--- id | tipo=int64 | valores únicos en muestra=1000
Ejemplos: [71079, 80590, 170711]

--- battle_time | tipo=object | valores únicos en muestra=190
Ejemplos: ['20261119T090001.000Z', '20261119T090001.000Z', '20261119T090001.000Z']

--- mode | tipo=object | valores únicos en muestra=6
Ejemplos: ['heist', 'brawlBall', 'hotZone']

--- map | tipo=object | valores únicos en muestra=24
Ejemplos: ['Kaboom Canyon', 'Triple Dribble', 'Dueling Beetles']

--- record | tipo=object | valores únicos en muestra=12
Ejemplos: ['T2-T2', 'T1-T2-T1', 'T1-T1']

--- star_brawler | tipo=object | valores únicos en muestra=94
Ejemplos: ['MELODIE', 'RICO', 'KENJI']

--- star_power | tipo=int64 | valores únicos en muestra=1
Ejemplos: [11, 11, 11]

--- star_player_tag | tipo=object | valores únicos en muestra=999
Ejemplos: ['#2LV0VPJ0VG', '#PR2VGG9Q9', '#2G9992GG9R']

--- star_elo | tipo=int64 | valores únicos en muestra=11
Ejemplos: [16, 15, 14]

--- avg_elo | tipo=float64 | valores únicos en muestra=42
Ejempl

Ahora, para las columnas con pocos valores distintos (mapa, modo, resultado, etc.), vemos la distribución completa.

In [9]:
for col in df_sample.columns:
    serie = df_sample[col].astype(str)
    if 1 < serie.nunique() <= 30:
        print(f"\n=== {col} ===")
        print(serie.value_counts().head(15))


=== mode ===
mode
knockout     181
brawlBall    179
gemGrab      169
hotZone      161
heist        155
bounty       155
Name: count, dtype: int64

=== map ===
map
Flaring Phoenix    57
Pinball Dreams     49
Bridge Too Far     47
Ring of Fire       47
Belle's Rock       46
Sneaky Fields      45
Hideout            44
Kaboom Canyon      44
Undermine          44
Double Swoosh      43
Triple Dribble     43
Gem Fort           42
Center Stage       42
Dry Season         41
Parallel Plays     41
Name: count, dtype: int64

=== record ===
record
T1-T1       303
T2-T2       300
T2-T1-T1     78
T1-T2-T1     75
T1-T2-T2     74
T2-T1-T2     73
T2           50
T1           42
T1-D-T2       2
D-T1-T1       1
D-T2-T1       1
T2-D-T2       1
Name: count, dtype: int64

=== star_elo ===
star_elo
14    365
13    251
15    147
12    109
16     69
17     33
18     11
11      6
19      6
10      2
20      1
Name: count, dtype: int64

=== t1_b0_elo ===
t1_b0_elo
14    347
13    255
15    146
12    116
16     

Si la columna de equipos viene como lista, la expandimos para ver cuántos brawlers hay por equipo y si siempre son 3.

In [10]:
listas = [c for c in df_sample.columns if df_sample[c].apply(lambda x: isinstance(x, (list, tuple))).any()]
print("Columnas tipo lista:", listas)

for c in listas:
    longitudes = df_sample[c].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else None)
    print(f"\n{c}: longitud por fila ->", longitudes.value_counts(dropna=False).to_dict())
    print("Ejemplo:", df_sample[c].iloc[0])

Columnas tipo lista: []


## 6. Volumen de datos y uso de memoria
Calculamos cuántas filas hay (leyendo solo los metadatos del dataset) y cuánta memoria ocupa una fila promedio, para estimar si cabe en Colab.

In [11]:
from datasets import load_dataset_builder

try:
    builder = load_dataset_builder(DATASET_ID)
    split_info = builder.info.splits.get("train")
    n_filas = split_info.num_examples if split_info else None
    print("Filas totales (según metadatos):", n_filas)
except Exception as e:
    n_filas = None
    print("No se pudieron leer los metadatos de filas:", e)

bytes_por_fila = df_sample.memory_usage(deep=True).sum() / len(df_sample)
print(f"Memoria promedio por fila en pandas: {bytes_por_fila:.0f} bytes")

if n_filas:
    print(f"Estimación para el dataset completo en pandas: {bytes_por_fila * n_filas / 1e9:.2f} GB")
    print("Colab estándar ofrece aproximadamente 12 GB de RAM; revisa en tu sesión con !free -h")

Filas totales (según metadatos): 2324251
Memoria promedio por fila en pandas: 1176 bytes
Estimación para el dataset completo en pandas: 2.73 GB
Colab estándar ofrece aproximadamente 12 GB de RAM; revisa en tu sesión con !free -h


### Recorrido completo (opcional)
Solo se ejecuta si `FULL_SCAN = True`. Cuenta las filas recorriendo el stream y mide el tiempo, para documentar el costo real.

In [12]:
if FULL_SCAN:
    inicio = time.time()
    total = 0
    for _ in load_dataset(DATASET_ID, split="train", streaming=True):
        total += 1
    print(f"Filas contadas: {total:,} en {time.time() - inicio:.1f} s")
else:
    print("FULL_SCAN desactivado. Cambia la constante a True si necesitas el conteo exacto.")

FULL_SCAN desactivado. Cambia la constante a True si necesitas el conteo exacto.


## 7. Resumen para el informe
Completa esta tabla con los valores que imprimió el notebook. No copies valores de memoria: deben salir de tu ejecución.

In [13]:
resumen = {
    "BrawlAPI: brawlers devueltos": len(brawlers),
    "BrawlAPI: campo de imagen usado": IMG_FIELD,
    "Imágenes: formato(s)": df_img["formato"].dropna().unique().tolist() if "formato" in df_img else "ver error",
    "Imágenes: resolución(es)": df_img[["ancho", "alto"]].drop_duplicates().values.tolist() if "ancho" in df_img else "ver error",
    "Dataset: columnas": df_sample.columns.tolist(),
    "Dataset: filas totales (metadatos)": n_filas,
    "Dataset: memoria promedio por fila (bytes)": round(bytes_por_fila),
    "Dataset: muestra inspeccionada": len(df_sample),
}
for k, v in resumen.items():
    print(f"{k}: {v}")

BrawlAPI: brawlers devueltos: 109
BrawlAPI: campo de imagen usado: imageUrl
Imágenes: formato(s): ['PNG']
Imágenes: resolución(es): [[200, 200]]
Dataset: columnas: ['id', 'battle_time', 'mode', 'map', 'record', 'star_brawler', 'star_power', 'star_player_tag', 'star_elo', 'avg_elo', 't1_b0_name', 't1_b0_elo', 't1_b0_power', 't1_b0_tag', 't1_b1_name', 't1_b1_elo', 't1_b1_power', 't1_b1_tag', 't1_b2_name', 't1_b2_elo', 't1_b2_power', 't1_b2_tag', 't2_b0_name', 't2_b0_elo', 't2_b0_power', 't2_b0_tag', 't2_b1_name', 't2_b1_elo', 't2_b1_power', 't2_b1_tag', 't2_b2_name', 't2_b2_elo', 't2_b2_power', 't2_b2_tag', 'skill_ns', 'skill_ns_ok']
Dataset: filas totales (metadatos): 2324251
Dataset: memoria promedio por fila (bytes): 1176
Dataset: muestra inspeccionada: 1000
